# Task 7 — Depth Approximation from Two Images

**Member 4** | Estimate depth using stereo disparity.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from cv_utils import load_image, show_images_row, to_grayscale
from tasks.task7_depth_approximation import (
    compute_disparity_bm, compute_disparity_sgbm,
    normalize_disparity, create_depth_map, create_colored_depth
)

print("✅ Imports OK")

## 1. Load Stereo Pair

Take two photos of the same scene, shifting the camera slightly to the right (~5-10cm).

In [ ]:
left = load_image('../images/task7/left.jpg')   # <-- CHANGE THIS
right = load_image('../images/task7/right.jpg')  # <-- CHANGE THIS

show_images_row([left, right], ["Left View", "Right View"], figsize=(16, 6))

## 2. Compute Disparity Maps

In [ ]:
left_gray = to_grayscale(left)
right_gray = to_grayscale(right)

# Method 1: StereoBM
disp_bm = compute_disparity_bm(left_gray, right_gray, num_disparities=64, block_size=15)
depth_bm = create_depth_map(disp_bm, smooth=True)

# Method 2: StereoSGBM (better quality)
disp_sgbm = compute_disparity_sgbm(left_gray, right_gray, num_disparities=64, block_size=7)
depth_sgbm = create_depth_map(disp_sgbm, smooth=True)

print("Disparity computed!")

## 3. Visualize Depth Maps

In [ ]:
depth_bm_color = create_colored_depth(depth_bm)
depth_sgbm_color = create_colored_depth(depth_sgbm)

show_images_row(
    [left, depth_bm, depth_bm_color, depth_sgbm_color],
    ["Original", "Depth BM (gray)", "Depth BM (color)", "Depth SGBM (color)"],
    figsize=(24, 6)
)

## 4. Tune Parameters

Adjust `numDisparities` and `blockSize` for best results.

In [ ]:
# Try different settings
for nd in [32, 64, 128]:
    for bs in [9, 15, 21]:
        disp = compute_disparity_bm(left_gray, right_gray, num_disparities=nd, block_size=bs)
        depth = create_depth_map(disp)
        plt.figure(figsize=(6, 4))
        plt.imshow(depth, cmap='jet')
        plt.title(f"numDisp={nd}, blockSize={bs}")
        plt.colorbar()
        plt.axis('off')
        plt.show()

## 5. Save Results

In [ ]:
os.makedirs('../outputs/task7', exist_ok=True)
cv2.imwrite('../outputs/task7/depth_bm.png', depth_bm)
cv2.imwrite('../outputs/task7/depth_bm_color.png', depth_bm_color)
cv2.imwrite('../outputs/task7/depth_sgbm_color.png', depth_sgbm_color)
print("✅ Saved to outputs/task7/")